### XGBoost tuning using a custom two-parameter scorer


ChatGPT couldn't solve this problem - so I had to tune it manually.
<br><br>My prompt was:
<br><font face="Courier" color="#3333cc">
<br>Use XGBoost regressor with sklearn interface.
<br>
<br>Generate 10,000 rows of synthetic data with 5 numeric features 
<br>and add correlation between those features and the target column.
<br>
<br>Write python code to demonstrate the use of make_scorer from sklearn.metrics.
<br>
<br>Create a custom estimator function to evaluate combination of two results:
<br>1. MSE (model error)
<br>2. The size of model file
<br>
<br>Use parameter grid like this:
<br>
<br>param_grid = {
<br>   'max_depth': [3, 5, 7],
<br>   'learning_rate': [0.1, 0.01, 0.001],
<br>   'subsample': [0.5, 0.7, 1],
<br>   'n_estimators': [50, 100, 200]
<br>}
<br>
<br>Make a 3D plot showing the score as function of MSE and size of the model
</font>

<br>Unfortunately it was giving errors that some parameters were missing.
<br>So after several several attempts I had to dive into the code and fix it myself.
<br>I couldn't figure out how to use the GridSearchCV 
<br> with the combined scorer (make_scorer),
<br> so I simply wrote a for-loop myself with a simple sccatter plot

In [1]:
import os, pickle
import numpy as np
import xgboost as xgb
from sklearn.datasets import make_regression
from sklearn.metrics import make_scorer, mean_squared_error
# from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from joblib import dump
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
# Generate synthetic data with 5 numeric features 
# and add correlation between those features and the target column
X, y = make_regression(n_samples=10000, n_features=5, noise=0.1, random_state=42)
for i in range(X.shape[1]):
    X[:, i] *= np.random.uniform(0.5, 1.5)
    y += 0.2 * X[:, i]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# define the grid of parameters we will be testing
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.1, 0.01, 0.001],
    'subsample': [0.5, 0.7, 1],
    'n_estimators': [50, 100, 200]
}

In [4]:
# Use make_scorer to create a custom scoring function based on mean squared error and model size
def model_size(estimator):
    dump(estimator, 'model.joblib')  # save the trained model to disk
    return -np.log10(os.path.getsize('model.joblib'))  # return -log10(model size in bytes)

In [17]:
results = {}
ii=0
for max_depth in param_grid['max_depth']:
    for learning_rate in param_grid['learning_rate']:
        for subsample in param_grid['subsample']:
            for n_estimators in param_grid['n_estimators']:
                ii += 1
                xgb_model = xgb.XGBRegressor(objective='reg:squarederror', 
                                             random_state=55,
                                             max_depth=max_depth,
                                             learning_rate=learning_rate,
                                             subsample=subsample,
                                             n_estimators=n_estimators
                                            )
                ss = f"max_depth:{max_depth},learn_rate:{learning_rate},subsample:{subsample},n_estim:{n_estimators}"
                # print(f"training {ss}",)
                xgb_model.fit(X_train, y_train)
                y_pred = xgb_model.predict(X_test)
                mse = round(mean_squared_error(y_test, y_pred),3)
                # r2 = round(r2_score(y_test, y_pred),3)
                fname = 'junk.pkl'
                with open(fname, 'wb') as f:
                    pickle.dump(xgb_model, f)
                fsize_kb = round(os.path.getsize(fname) / 1024.0)
                results[ss] = {'mse':mse,'kb':fsize_kb}
                print(f"{ii:3} : {str(ss):25}: {str(results[ss]):30}")

  1 : max_depth:3,learn_rate:0.1,subsample:0.5,n_estim:50: {'mse': 328.884, 'kb': 62}    
  2 : max_depth:3,learn_rate:0.1,subsample:0.5,n_estim:100: {'mse': 87.421, 'kb': 118}    
  3 : max_depth:3,learn_rate:0.1,subsample:0.5,n_estim:200: {'mse': 58.844, 'kb': 226}    
  4 : max_depth:3,learn_rate:0.1,subsample:0.7,n_estim:50: {'mse': 322.934, 'kb': 62}    
  5 : max_depth:3,learn_rate:0.1,subsample:0.7,n_estim:100: {'mse': 81.373, 'kb': 118}    
  6 : max_depth:3,learn_rate:0.1,subsample:0.7,n_estim:200: {'mse': 51.871, 'kb': 229}    
  7 : max_depth:3,learn_rate:0.1,subsample:1,n_estim:50: {'mse': 323.955, 'kb': 62}    
  8 : max_depth:3,learn_rate:0.1,subsample:1,n_estim:100: {'mse': 78.31, 'kb': 118}     
  9 : max_depth:3,learn_rate:0.1,subsample:1,n_estim:200: {'mse': 50.45, 'kb': 228}     
 10 : max_depth:3,learn_rate:0.01,subsample:0.5,n_estim:50: {'mse': 10655.345, 'kb': 62}  
 11 : max_depth:3,learn_rate:0.01,subsample:0.5,n_estim:100: {'mse': 6322.737, 'kb': 119}  
 12 : m

In [18]:
# Just looking at it I see mse ranging from ~50 to 18,000
# Let us only consider entries where mse < 100
res2 = {}
for k,v in results.items():
    mse = v['mse']
    kb = v['kb']
    if mse >= 100:
        continue
    res2[k] = v
    print(f"{k} => {v}")

max_depth:3,learn_rate:0.1,subsample:0.5,n_estim:100 => {'mse': 87.421, 'kb': 118}
max_depth:3,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 58.844, 'kb': 226}
max_depth:3,learn_rate:0.1,subsample:0.7,n_estim:100 => {'mse': 81.373, 'kb': 118}
max_depth:3,learn_rate:0.1,subsample:0.7,n_estim:200 => {'mse': 51.871, 'kb': 229}
max_depth:3,learn_rate:0.1,subsample:1,n_estim:100 => {'mse': 78.31, 'kb': 118}
max_depth:3,learn_rate:0.1,subsample:1,n_estim:200 => {'mse': 50.45, 'kb': 228}
max_depth:5,learn_rate:0.1,subsample:0.5,n_estim:100 => {'mse': 64.202, 'kb': 241}
max_depth:5,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 48.763, 'kb': 465}
max_depth:5,learn_rate:0.1,subsample:0.7,n_estim:100 => {'mse': 59.96, 'kb': 249}
max_depth:5,learn_rate:0.1,subsample:0.7,n_estim:200 => {'mse': 48.484, 'kb': 481}
max_depth:5,learn_rate:0.1,subsample:1,n_estim:100 => {'mse': 64.68, 'kb': 259}
max_depth:5,learn_rate:0.1,subsample:1,n_estim:200 => {'mse': 55.432, 'kb': 474}
max_depth:7,lear

In [19]:
# Hmm, looks like all of these have learning_rate = 0.1. Interesting.
# Let us now look only at mse < 60:
res3 = {}
for k,v in res2.items():
    mse = v['mse']
    kb = v['kb']
    if mse >= 60:
        continue
    res3[k] = v
    print(f"{k} => {v}")

max_depth:3,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 58.844, 'kb': 226}
max_depth:3,learn_rate:0.1,subsample:0.7,n_estim:200 => {'mse': 51.871, 'kb': 229}
max_depth:3,learn_rate:0.1,subsample:1,n_estim:200 => {'mse': 50.45, 'kb': 228}
max_depth:5,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 48.763, 'kb': 465}
max_depth:5,learn_rate:0.1,subsample:0.7,n_estim:100 => {'mse': 59.96, 'kb': 249}
max_depth:5,learn_rate:0.1,subsample:0.7,n_estim:200 => {'mse': 48.484, 'kb': 481}
max_depth:5,learn_rate:0.1,subsample:1,n_estim:200 => {'mse': 55.432, 'kb': 474}
max_depth:7,learn_rate:0.1,subsample:0.5,n_estim:100 => {'mse': 59.816, 'kb': 512}
max_depth:7,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 52.395, 'kb': 1014}
max_depth:7,learn_rate:0.1,subsample:0.7,n_estim:200 => {'mse': 54.677, 'kb': 1119}


In [20]:
# Looks like the best number of trees (n_estimators) is big (around 200)
# Also looks like training trees only on a fraction of training data (subsample <1) is OK
# But it produces much smaller model.
# So let's use subsample = 0.5 and n_estimators = 200

def parse_key(ss):
    dd = {}
    for part in ss.split(","):
        kk,vv = part.split(":")
        dd[kk] = float(vv)
    return dd

res4 = {}
for k,v in res3.items():
    mse = v['mse']
    kb = v['kb']
    if mse >= 56:
        continue
    dd = parse_key(k)
    # print(dd)
    if dd['subsample'] > 0.6:
        continue
    if dd['n_estim'] < 150:
        continue
    res4[k] = v
    print(f"{k} => {v}")

max_depth:5,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 48.763, 'kb': 465}
max_depth:7,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 52.395, 'kb': 1014}


In [21]:
# OK, selecting this set of parameters:
# max_depth:5,learn_rate:0.1,subsample:0.5,n_estim:200 => {'mse': 48.763, 'kb': 465}